# Session 2.4 — Production Training Pipeline

## Theory: Why Do We Need a Dataset Class?

### The Core Problem

Right now, your data lives as files on disk. Your model lives in memory. These two worlds need a bridge.

Imagine you're training on 10,000 audio samples. You *could* load everything into memory at once — but that's wasteful, slow to start, and often impossible at scale. Instead, PyTorch uses a **lazy loading** pattern: load only what you need, when you need it.

That's exactly what a `Dataset` class does.

---

### The Abstraction: What is a Dataset?

Think of a PyTorch `Dataset` like an **index card system at a library.**

- The librarian (DataLoader) asks: *"Give me card #347"*
- The Dataset knows where card #347 is stored and how to retrieve it
- It returns the card in a standard format the model can read

Your Dataset doesn't need to know about batching, shuffling, or parallelism. It just needs to answer two questions:

1. **"How many items do you have?"** → `__len__()`
2. **"Give me item number N"** → `__getitem__(n)`

That's the entire contract.

---

### The Three Responsibilities of Your Dataset

```
__init__()       → "Set up the index" (load manifest, store paths)
__len__()        → "How big is the library?"
__getitem__()    → "Fetch one book by index"
```

**Critical insight:** `__init__` should be *fast and cheap*. Don't load audio there. Don't compute features there. Just read the manifest and store the metadata. All the heavy work happens in `__getitem__`, which PyTorch calls lazily — only when it actually needs that sample.

---

### What Lives in Your Manifest?

Your manifest (from Phase 1) is a JSON file where each entry describes one sample. At minimum, you need:
- The path to the audio file
- The number of speakers (your label)

When `__getitem__` is called with index `i`, it should:
1. Look up entry `i` in the manifest
2. Load the audio from disk
3. Compute your 5 features (mean, std, min, max, energy)
4. Convert everything to tensors
5. Return `(features_tensor, label_tensor)`

---

### Label Encoding

Your model outputs 3 classes. Your labels are speaker counts (1, 2, or 3). PyTorch's CrossEntropyLoss expects class indices starting from **0**.

So: 1 speaker → class 0, 2 speakers → class 1, 3 speakers → class 2.

The transform is simply: `label = num_speakers - 1`

---

## TODO 1: Custom Dataset Class

**Goal:** Build a `AudioMixtureDataset` class that PyTorch's DataLoader can use to feed samples into your model one-by-one (or in batches).

**Requirements:**
- Inherits from `torch.utils.data.Dataset`
- `__init__` takes a path to your manifest JSON and loads it into memory as a list
- `__len__` returns the number of samples
- `__getitem__` takes an integer index and returns a tuple of two tensors: `(features, label)`
- Features tensor shape: `(5,)` — your 5 hand-crafted features
- Label tensor: a single integer class index (0, 1, or 2), dtype `torch.long`
- Feature extraction logic should live *inside* `__getitem__`, not `__init__`

**Hints:**
- Import `json`, `torch`, and whatever you used in Phase 1 for audio loading (likely `torchaudio` or `librosa`)
- `json.load(f)` on an open file gives you a Python list/dict — explore your manifest first to know its structure
- For the label: `num_speakers - 1` converts to zero-indexed class
- `torch.tensor(value, dtype=torch.long)` for the label
- `torch.tensor(your_list, dtype=torch.float32)` for features
- Check your manifest structure first by printing one entry in a notebook — don't guess the key names

**Before you write a single line of code**, open your manifest and answer these questions:
1. Is it a list of dicts, or a dict of lists?
2. What key holds the audio file path?
3. What key holds the number of speakers?

**After it works — EXPERIMENTS:**
- Instantiate the dataset and print its length
- Fetch `dataset[0]`, `dataset[100]`, `dataset[999]` — do they all have shape `(5,)` for features?
- Print the unique label values across 20 random samples — are you seeing 0, 1, and 2?
- Time how long `__getitem__` takes for a single sample (use Python's `time` module)

---

Take your time exploring the manifest first, then implement. Let me know what structure you find in your manifest and what you've tried — I'll be here when you have questions or hit a wall.

In [290]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from typing import List, Tuple, Dict
from pathlib import Path
import os
import sys
import json
import random

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


In [291]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            self.data = json.load(f)
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = mixture_tensor
        feature_tensor = self.feature_extraction(mixture_tensor)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return feature_tensor, label_tensor
    
    def feature_extraction(
            self,
            audio: torch.Tensor
    ) -> torch.Tensor:
        mean = torch.mean(audio)
        std = torch.std(audio)
        min = torch.min(audio)
        max = torch.max(audio)
        square = audio ** 2
        energy = torch.mean(square)
        
        return torch.stack([mean, std, min, max, energy])

In [292]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path)
print(len(val_dataset))


10000
10000


In [293]:
features, label = train_dataset[0]
print(features.shape, label)  # expect torch.Size([5]), 0 or 1 or 2


torch.Size([5]) tensor(2)


In [294]:
features, label = train_dataset[500]
print(features.shape, label) 

torch.Size([5]) tensor(2)


In [295]:
features, label = train_dataset[9999]
print(features.shape, label) 

torch.Size([5]) tensor(1)


In [296]:
for i in range(0,5):
    _, label = train_dataset[i]
    print(f"{i}th label is {label}")

0th label is 2
1th label is 2
2th label is 1
3th label is 2
4th label is 1


## Part 2 — DataLoaders

### Theory: What Does DataLoader Actually Do?

Your `SpeakerDataset` knows how to fetch *one* sample. But your model trains on *batches*. DataLoader is the bridge — it calls `__getitem__` repeatedly, stacks the results into batches, and optionally shuffles the order each epoch.

The key parameter to understand is **`shuffle`**:
- Train loader: `shuffle=True` — different order every epoch, prevents the model memorizing order
- Val loader: `shuffle=False` — consistent order, reproducible evaluation

---

## TODO 2: Create DataLoaders

**Goal:** Create train and val DataLoaders from your dataset.

**Requirements:**
- Create `SpeakerDataset` instances for both train and val manifests
- Val manifest path follows the same pattern as train (swap `train` for `val`)
- `train_loader`: `batch_size=64`, `shuffle=True`, `num_workers=0`
- `val_loader`: `batch_size=64`, `shuffle=False`, `num_workers=0`
- Iterate 3 batches from `train_loader` and print the shape of features and labels each time

**Expected output:**
```
Batch 0: features=torch.Size([64, 5])  labels=torch.Size([64])
Batch 1: features=torch.Size([64, 5])  labels=torch.Size([64])
Batch 2: features=torch.Size([64, 5])  labels=torch.Size([64])
```

**Hints:**
- `DataLoader` is already imported in your notebook
- To iterate only 3 batches, look at `enumerate()` and a conditional `break`
- The last batch of an epoch may be smaller than 64 if dataset size isn't divisible — there's a parameter to handle this if you want clean batches

Give it a go and show me your output when done.

In [297]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

In [298]:
for i,j in enumerate(train_loader):
    print(f"Batch {i}: features={j[0].shape}  labels={j[1].shape}")
    if i == 2:
        break

Batch 0: features=torch.Size([64, 5])  labels=torch.Size([64])
Batch 1: features=torch.Size([64, 5])  labels=torch.Size([64])
Batch 2: features=torch.Size([64, 5])  labels=torch.Size([64])


## TODO 3: Verify Epoch Coverage

**Goal:** Prove that one full pass through the DataLoader sees every sample exactly once.

**Requirements:**
- Temporarily modify `__getitem__` to return the index `idx` as a third value
- Iterate one full epoch of `train_loader`, collecting all indices seen
- Assert that the set of collected indices equals `{0, 1, ..., len(train_dataset)-1}`
- Print a confirmation message if it passes

**Hints:**
- DataLoader will return whatever `__getitem__` returns — adding a third item to the tuple is fine temporarily
- A Python `set` is useful for checking coverage
- Remember `drop_last=True` means your coverage check needs to account for the dropped samples



In [299]:
class SpeakerDatasetM1(torch.utils.data.Dataset):
    def __init__(self, manifest_path):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            self.data = json.load(f)
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = mixture_tensor
        feature_tensor = self.feature_extraction(mixture_tensor)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return feature_tensor, label_tensor, idx
    
    def feature_extraction(
            self,
            audio: torch.Tensor
    ) -> torch.Tensor:
        mean = torch.mean(audio)
        std = torch.std(audio)
        min = torch.min(audio)
        max = torch.max(audio)
        square = audio ** 2
        energy = torch.mean(square)
        
        return torch.stack([mean, std, min, max, energy])

In [300]:
train_dataset_M1 = SpeakerDatasetM1(train_manifest_path)

batch_size = 64

train_loader_M1 = DataLoader(
    dataset=train_dataset_M1,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=False
)


collected_indices = set()

for batch in train_loader_M1:
    _, _, idx = batch
    collected_indices.update(idx.tolist())

assert len(collected_indices) == len(train_dataset_M1), f"Coverage mismatch: seen {len(collected_indices)} indices, expected { len(train_dataset_M1) }"
assert collected_indices.issubset(set(range(len(train_dataset_M1)))), "Indices found outside of dataset range"

## TODO 4 (revised): Full Training + Validation Loop

**Requirements:**
- Outer loop: epochs
- Inner loop 1 (train): forward, loss, backward, step — track average train loss
- Inner loop 2 (val): forward only, track average val loss AND accuracy
- After each epoch print: `Epoch N | Train Loss: X.XXX | Val Loss: X.XXX | Val Accuracy: XX.X%`

**Accuracy hint:** Your model outputs shape `(batch_size, 3)` — raw scores per class. To get the predicted class, you need the index of the highest score. There's a PyTorch function that returns the maximum value and its index along a dimension. Then compare predictions to true labels and count matches.

**Structure hint:**
``` t
correct = 0
total = 0
# after each batch in val loop:
# get predicted class from outputs
# compare to labels
# accumulate correct and total
accuracy = correct / total
```

In [301]:
class SpeakerCounter(nn.Module):
    """
    Classifier that predicts number of speakers (1, 2, or 3)
    from audio features.
    """
    def __init__(self, input_size=5, hidden_sizes=[64, 32, 16], num_classes=3):
        super().__init__()
        # TODO: Build network with given architecture
        # Hint: Use nn.Sequential or define layers individually
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_sizes[0]),
            nn.BatchNorm1d(hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_sizes[0],hidden_sizes[1]),
            nn.BatchNorm1d(hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_sizes[1],hidden_sizes[2]),
            nn.BatchNorm1d(hidden_sizes[2]),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_sizes[2],num_classes)
        )
        
    def forward(self, x):
        # TODO: Implement forward pass
        return self.network(x)

In [302]:
def compute_accuracy(predictions, labels):
    """
    predictions: (batch_size, 3) logits
    labels: (batch_size,) true classes
    
    Returns: accuracy as percentage
    """
    predicted_classes = torch.argmax(predictions, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total

In [303]:
import torch.optim as optim
import torch.nn.functional as F


model = SpeakerCounter().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_accuracy = 0.0
    steps_per_epoch = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = model(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        optimizer.zero_grad()
        
        # Compute accuracy
        running_loss += loss.item() 
        running_accuracy += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_epoch_loss = running_loss / steps_per_epoch
    avg_epoch_train_acc = running_accuracy / steps_per_epoch
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Avg Train Loss: {avg_epoch_loss:.4f} | Avg Train Acc: {avg_epoch_train_acc:.2f}%")  
    
    
    model.eval()
    val_accuracy_sum = 0.0
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            val_accuracy_sum += compute_accuracy(logits, labels)
            
            # total sample in the batch
            total += len(labels)
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (step + 1) % 50 == 0:
                print(f"  Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    avg_val_acc = val_accuracy_sum / len(val_loader)
    print(f"Average Validation Accuracy: {avg_val_acc:.2f}%\n")




  Batch 50/156 | Loss: 1.2312
  Batch 100/156 | Loss: 1.1580
  Batch 150/156 | Loss: 1.1783

--- Epoch 1/5 Summary ---
Avg Train Loss: 1.2149 | Avg Train Acc: 26.84%
Average Validation Accuracy: 47.77%

  Batch 50/156 | Loss: 1.1634
  Batch 100/156 | Loss: 1.1115
  Batch 150/156 | Loss: 1.1138

--- Epoch 2/5 Summary ---
Avg Train Loss: 1.1185 | Avg Train Acc: 35.36%
Average Validation Accuracy: 51.50%

  Batch 50/156 | Loss: 1.0755
  Batch 100/156 | Loss: 1.1010
  Batch 150/156 | Loss: 1.0687

--- Epoch 3/5 Summary ---
Avg Train Loss: 1.0478 | Avg Train Acc: 44.54%
Average Validation Accuracy: 52.27%

  Batch 50/156 | Loss: 0.9865
  Batch 100/156 | Loss: 0.9692
  Batch 150/156 | Loss: 0.9944

--- Epoch 4/5 Summary ---
Avg Train Loss: 0.9976 | Avg Train Acc: 49.54%
Average Validation Accuracy: 53.84%

  Batch 50/156 | Loss: 0.9568
  Batch 100/156 | Loss: 0.9362
  Batch 150/156 | Loss: 0.9427

--- Epoch 5/5 Summary ---
Avg Train Loss: 0.9529 | Avg Train Acc: 51.38%
Average Validation Acc